In [ ]:
# Install core PyTorch (if not already handled by your RunPod template)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Install standard Hugging Face stack and Flash Attention
pip install transformers peft trl datasets accelerate bitsandbytes human-eval xformers flash-attn --no-build-isolation

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import os

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
max_seq_length = 2048 

# 1. Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Qwen models often need the pad token explicitly set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Setup 4-bit Quantization (BitsAndBytes)
# Since you have a 24GB GPU, a 1.5B model easily fits natively. 
# But we'll keep 4-bit enabled to match your previous setup and save maximum VRAM.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load the Model with Flash Attention 2
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # <--- The industry standard for speed
)

# Prepare model for 4-bit training (handles layer freezing and gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# 4. Apply LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=16, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 5. Format the Prompts
prompt_template = """Below is an instruction that describes a programming task. Write a response that appropriately completes the request.
### Instruction:
{prompt}
### Response:
{response}"""

def format_prompts(examples):
    prompts = examples["prompt"]
    # If long_cot exists, use it. Otherwise, use short_cot.
    responses = [l if l else s for l, s in zip(examples.get("long_cot", [""]*len(prompts)), examples["short_cot"])]
    
    texts = []
    for p, r in zip(prompts, responses):
        texts.append(prompt_template.format(prompt=p, response=r))
    return { "text" : texts }

# 6. Execute the Curriculum
epochs_data = [
    "/workspace/dataset/epoch_1.jsonl",
    "/workspace/dataset/epoch_2.jsonl",
    "/workspace/dataset/epoch_3.jsonl"
]

for epoch_idx, dataset_path in enumerate(epochs_data):
    print(f"\n{'='*40}")
    print(f"STARTING CURRICULUM EPOCH {epoch_idx + 1}")
    print(f"{'='*40}\n")
    
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(format_prompts, batched=True)
    
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            dataset_num_proc=2,
            padding_free=False, 
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=1, 
            learning_rate=2e-4,
            fp16=False,
            bf16=True, # Modern GPUs should always use bf16
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            output_dir=f"outputs_epoch_{epoch_idx+1}",
            report_to="none" 
        ),
    )
    
    trainer.train()
    
    # Save the adapter weights after each epoch
    save_path = f"lora_model_epoch_{epoch_idx+1}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Checkpoint saved to {save_path}")

print("Training Complete! All epochs finished.")